In [26]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Đọc tệp dữ liệu Cancer.csv trong file data_cancer
df = pd.read_csv('D:/Teach_n_Train/Machine Learning/exam_n_practice/test_NB/food_order.csv')

print(df.head() )


         cuisine     city       device user_segment    promo fav_ingredient  \
0       japanese  city_19          web      student  promo_8         sesame   
1        chinese   city_1          ios       budget  promo_5            soy   
2  mediterranean  city_14  desktop_app       family  promo_8         yogurt   
3        italian  city_15       tablet          new  promo_4          lemon   
4     vietnamese  city_10          ios          new  promo_8          basil   

  spice_level day_of_week price_bucket  
0      medium         Tue          mid  
1   extra_hot         Wed    very_high  
2        none         Mon          mid  
3        none         Sat          mid  
4        mild         Tue          mid  


###### Tính các xác suất p_k := N_k / N (chú ý N = len(df) )

In [27]:
target_feature = 'cuisine'

Y = df[target_feature] 
labels = np.zeros((len(set(Y)), 2), dtype=Y.dtype) 

# Liệt kê các nhãn (label) khác nhau (từ trường cuisine) k =1, 2, ... C
# và tính xác suất P_k của mỗi nhãn. Sử dụng nhãn thay cho chỉ số
id = 0
for label in set(Y):
    labels[id, 0] = label
    labels[id, 1] = (Y == label).sum()/len(Y)
    id += 1

###### Lấy tên mỗi trường (field) của dữ liệu và số lựa chọn ứng với trường đó

In [28]:
num_labels_M = np.array([['cuisine', 10],
                         ['city', 20],
                         ['device', 5],
                         ['user_segment', 6],
                         ['promo', 9],
                         ['fav_ingredient', 18],
                         ['spice_level', 5],
                         ['day_of_week', 7],
                         ['price_bucket', 4]] )

print(num_labels_M)

[['cuisine' '10']
 ['city' '20']
 ['device' '5']
 ['user_segment' '6']
 ['promo' '9']
 ['fav_ingredient' '18']
 ['spice_level' '5']
 ['day_of_week' '7']
 ['price_bucket' '4']]


###### Tính các xác suất thành phần P(xi | k) theo công thức xấp xỉ (qua Smooth Laplacian) P(xi| y = k) ~ lambda_ik := [|{xm in class k : x_im = xi}| + alpha] / [N_k + M*alpha]

In [29]:
def Pxik_feature_per_class(Xk, xi, alpha = 1.0):
# tính toán cho toàn bộ mảng thay vì dùng vòng lặp
    X_ = np.array(Xk.iloc[:, 1:])
    xi =  np.array(xi)
    Pxik = np.zeros(len(xi))
    M_nomials = (num_labels_M[1:, 1]).astype(int)
    N_k = len(Xk)
        
    count = (X_ == xi).sum()
    Pxik = (count + alpha)/(N_k + M_nomials*alpha)
    
#Returns the prob. of feature per class
    return np.array(Pxik, dtype=float)
    

###### Tính argmax_k { ln[P(k)*P(x|k)]} =  argmax_k { ln[P(k)]  + ln[P(x_1|k)] + ln[P(x_2|k)] + ... + ln[P(x_d|k)]}

In [30]:
def predict_output_label(X, x_input, target_feature):
    #score for each class Pk
    p = np.log(np.array(labels[:, 1], dtype=float))

    for k in range(len(p)):

        Xk = X[X[target_feature] == labels[k, 0]]

        P_xik = Pxik_feature_per_class(Xk, x_input)
        p[k] += np.sum(np.log(P_xik))
        
    y_star = np.argmax(p)
    return labels[y_star, 0]

In [39]:
Train_size = round(len(df)*0.5)
X_Train = df[:Train_size]
Y_Train = df[target_feature][:Train_size]

X_Test = df[Train_size:].drop(target_feature, axis = 1).reset_index(drop=True, inplace=False)
Y_Test = df[target_feature][Train_size:].reset_index(drop=True, inplace=False)

Y_pred = np.zeros(len(Y_Test), dtype = Y_Test.dtype) 
print(len(X_Test), len(Y_Test))

for i in range(len(X_Test)):
    Y_pred[i] = predict_output_label(X_Train, X_Test.iloc[i, :], target_feature)
    if(i % 50 == 0):
        print(i, Y_pred[i], Y_Test[i])

1500 1500
0 mexican mexican
50 chinese chinese
100 american vietnamese
150 vietnamese thai
200 chinese chinese
250 mexican mexican
300 chinese chinese
350 chinese indian
400 indian mexican
450 vietnamese thai
500 indian indian
550 japanese korean
600 chinese chinese
650 mexican japanese
700 italian chinese
750 thai thai
800 american american
850 thai italian
900 chinese japanese
950 indian indian
1000 american american
1050 korean japanese
1100 mediterranean italian
1150 thai indian
1200 mediterranean mexican
1250 chinese japanese
1300 american american
1350 mediterranean italian
1400 thai vietnamese
1450 vietnamese american


In [42]:
from sklearn.metrics import accuracy_score

print(accuracy_score(Y_pred, Y_Test))

0.4086666666666667


In [34]:
print(predict_output_label(X_Train, X_Train.iloc[3, 1:], target_feature) )

italian


In [41]:
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB

X_Train_n = X_Train.drop(target_feature, axis = 1).reset_index(drop=True, inplace=False)

le = LabelEncoder()
for col in X_Train_n.select_dtypes(include=['object', 'category']).columns:
    X_Train_n[col] = le.fit_transform(X_Train_n[col])


Y_Train = df[target_feature][:Train_size]

X_Test = df[Train_size:].drop(target_feature, axis = 1).reset_index(drop=True, inplace=False)
Y_Test = df[target_feature][Train_size:].reset_index(drop=True, inplace=False)

for col in X_Test.select_dtypes(include=['object', 'category']).columns:
    X_Test[col] = le.fit_transform(X_Test[col])

clf = MultinomialNB()

clf.fit(X_Train_n, Y_Train)
y_pred = clf.predict(X_Test)

print(accuracy_score(y_pred, Y_Test))

0.174
